# 00 -- What is a slot? (intro before everything else)

> **Companion repo**: this notebook inspects the
> [`chipathon-2026-gf180mcu-padring`](https://github.com/Mauricio-xx/chipathon-2026-gf180mcu-padring)
> fork (itself a derivation of
> [`wafer-space/gf180mcu-project-template`](https://github.com/wafer-space/gf180mcu-project-template)
> with a workshop slot mirroring
> [`JuanMoya/padring_gf180`](https://github.com/JuanMoya/padring_gf180)).
> The parser cell below clones the fork on demand.

# 00 — What is a slot? (intro before everything else)

Read this first. It is a short, read-only walkthrough of a single
concept that trips every first-time chipathon participant: the
**slot**. No flow runs here; when you are done, jump to the
numbered notebooks for the hands-on bits.

## TL;DR

A **slot** is *not* a LibreLane concept. It is a convention of the
wafer-space `gf180mcu-project-template` that groups together the
three things that change when you target a different die size on
the same multi-project shuttle:

1. How many pads of each kind you have (input / bidir / analog /
   power).
2. How big the die is, and where the standard-cell core sits inside
   the padring.
3. The exact clockwise order of pad instances around the padring.

LibreLane itself does not know about the word "slot". It just
accepts one or more YAML files on the command line and merges them.
The template splits that YAML in two so the stable parts
(`config.yaml`, with RTL, clock, PDN, macros) never change across
die sizes, and the die-specific parts (`slots/slot_<name>.yaml`,
with `DIE_AREA`, `CORE_AREA`, pad lists) do. That's all.

If your tape-out is not on wafer-space, you can flatten this and
put everything in one YAML. See `03_slot_create_workshop.ipynb`
for a realistic example of inventing your own.

## 1. The three files that make a slot

Every slot in the template is defined by three moving parts. You
cannot have a slot unless all three are in place and consistent:

### (a) `src/slot_defines.svh` — the pad counts

A Verilog-preprocessor header with one `\`ifdef SLOT_<NAME>` block
per slot. It declares five integers that `chip_top.sv` reads to
size its generate loops.

```verilog
`ifdef SLOT_WORKSHOP
`define NUM_DVDD_PADS  4    // power pads: VDD -> core
`define NUM_DVSS_PADS  4    // power pads: VSS -> core
`define NUM_INPUT_PADS 1    // pure digital inputs (no drive)
`define NUM_BIDIR_PADS 20   // digital bidir pads
`define NUM_ANALOG_PADS 60  // 5 V analog pads
`endif
```

### (b) `librelane/slots/slot_<name>.yaml` — the geometry + pad order

```yaml
FP_SIZING: absolute
DIE_AREA:  [0, 0, 2935, 2935]         # die outline (includes sealring)
CORE_AREA: [442, 442, 2493, 2493]     # std-cell rectangle inside padring
VERILOG_DEFINES: ["SLOT_WORKSHOP"]    # enables the matching block of (a)

PAD_SOUTH: [ clk_pad, rst_n_pad, "inputs\\[0\\].pad", ... ]
PAD_EAST:  [ ... ]
PAD_NORTH: [ ... ]
PAD_WEST:  [ ... ]
```

Pad lists read **clockwise starting from the south-west corner**:
PAD_SOUTH goes W -> E, PAD_EAST goes S -> N, PAD_NORTH goes E -> W,
PAD_WEST goes N -> S.

### (c) `Makefile` — the slot registration

A one-line whitelist that lets `make librelane SLOT=<name>` work:

```makefile
AVAILABLE_SLOTS = 1x1 0p5x1 1x0p5 0p5x0p5 workshop
```

That is *all* a slot is. No magic.

## 2. What changes across slots, what stays the same

Using the workshop-padring template (which ships with five slots),
here is the comparison:

In [ ]:
# This cell parses `src/slot_defines.svh` from the chipathon-2026
# padring fork to print a table of pad counts across slots.
# If the fork is not cloned yet, this cell clones it (~200 KB).

import re
import subprocess
from pathlib import Path

FORK_URL = 'https://github.com/Mauricio-xx/chipathon-2026-gf180mcu-padring.git'
TEMPLATE = Path.home() / 'eda' / 'designs' / 'chipathon_padring' / 'template'

if not TEMPLATE.exists():
    TEMPLATE.parent.mkdir(parents=True, exist_ok=True)
    print(f'Cloning fork into {TEMPLATE} ...')
    subprocess.run(['git', 'clone', '--depth', '1', FORK_URL, str(TEMPLATE)], check=True)

# Parse slot_defines.svh to extract pad counts per slot.
svh = (TEMPLATE / 'src' / 'slot_defines.svh').read_text()
slot_blocks = re.findall(r'`ifdef\s+SLOT_(\w+)(.*?)`endif', svh, re.DOTALL)

def pad_counts(block):
    counts = {}
    for m in re.finditer(r'`define\s+NUM_(\w+)_PADS\s+(\d+)', block):
        counts[m.group(1).lower()] = int(m.group(2))
    return counts

print(f'{"slot":>10}  {"dvdd":>5}  {"dvss":>5}  {"input":>5}  {"bidir":>5}  {"analog":>6}  total')
print('-' * 60)
for name, body in slot_blocks:
    c = pad_counts(body)
    total = sum(c.values())
    print(f'{name.lower():>10}  {c.get("dvdd",0):>5}  {c.get("dvss",0):>5}  {c.get("input",0):>5}  {c.get("bidir",0):>5}  {c.get("analog",0):>6}  {total:>5}')


Read the table: larger dies trade analog pads for digital bidir
and vice-versa, while power pad counts scale with total pad count
and core area.  `WORKSHOP` is the analog-heavy slot (60 analog, only
20 digital bidir), a mirror of JuanMoya's [padring_gf180](
https://github.com/JuanMoya/padring_gf180) config. The other four
slots are digital-heavy.

**What never changes across slots:** `src/chip_top.sv` (the padring
itself), `src/chip_core.sv` (your design), `librelane/config.yaml`
(clock, PDN, macros), `librelane/pdn_cfg.tcl` (PDN tcl). Exactly the
three pieces listed in section 1 are what a slot touches.

## 3. Pad arithmetic: will the padring actually close?

A padring closes only if on each side of the die:

$$\text{die\_side} \;\geq\; \sum \text{pad\_widths} \;+\; 2 \cdot \text{corner\_width} \;+\; \text{fillers}$$

For GF180MCU IO cells (`gf180mcu_fd_io__*`):

| cell                     | width   |
|--------------------------|---------|
| `asig_5p0` (analog)      | 75 um   |
| `bi_t`, `bi_24t` (bidir) | 75 um   |
| `dvdd`, `dvss`, `in_c`, `in_s` | 75 um |
| `cor` (corner)           | 355 um  |
| `fill10`, `fill5`, `fill1`, `fillnc` | 10 / 5 / 1 / 0.1 um |

Fillers are placed automatically. Your job in a new slot is only:

1. Pick a die side **D** (um).
2. Decide how many pads **N** go on each side.
3. Check that **D >= 75 N + 710** (the 710 um comes from the two
   corners shared with the neighbouring sides).  The remaining slack
   **D - 75 N - 710** is how much filler will be placed. Must be
   positive.

For the workshop slot (D = 2935, N = 22..25):

| side  | N  | pads (um) | corners | filler slack (um) |
|-------|----|-----------|---------|--------------------|
| S     | 25 | 1875      | 710     | 350                |
| E/N/W | 22 | 1650      | 710     | 575                |

Both positive => the padring closes. This is why we picked die 2935
for 90 total pads: it gives ~10-20% filler slack, healthy margin.

## 4. The clockwise-from-SW rule

LibreLane's `OpenROAD.Padring` step walks each `PAD_*` list in the
order you give it, placing the first entry nearest the south-west
corner of that side and the last entry nearest the next-clockwise
corner.

Visually (die centre is at +, corners marked with *):

```
         PAD_NORTH (entries read E -> W)
         * -------------------- *
         |                      |
  PAD_   |                      |   PAD_
  WEST   |           +          |   EAST
  (N->S) |                      |  (S->N)
         |                      |
         * -------------------- *
         PAD_SOUTH (entries read W -> E)
```

When translating a standalone-`padring`-tool cfg (like JuanMoya's)
into a LibreLane slot yaml, **NORTH and WEST lists must be reversed**
because the standalone tool uses "left-to-right" and "bottom-to-top"
per-side orientation. Get this wrong and `OpenROAD.Padring` fails
with "padring does not close".

## 5. Inside a `PAD_*` list — three name patterns

Entries in a `PAD_*` list are instance names in `chip_top.sv`.
Three shapes you will see:

In [ ]:
print("""
1. Hard-coded single instances (clk_pad and rst_n_pad in chip_top.sv):

       clk_pad
       rst_n_pad

2. Indexed instances from a generate loop. The generate label is
   the block name; [i] is the loop index; .pad is the instance
   name inside the block:

       \"analog\\\\[37\\\\].pad\"        # becomes: analog[37].pad  in the netlist
       \"bidir\\\\[11\\\\].pad\"         # becomes: bidir[11].pad
       \"dvdd_pads\\\\[2\\\\].pad\"      # becomes: dvdd_pads[2].pad
       \"dvss_pads\\\\[1\\\\].pad\"      # becomes: dvss_pads[1].pad
       \"inputs\\\\[0\\\\].pad\"         # becomes: inputs[0].pad

   The doubled backslashes escape the YAML string so LibreLane sees
   a literal '\\[' and '\\]' when the regex-matching it does against
   the flattened hierarchy. Without them the brackets are taken as
   regex metacharacters and the match breaks.

3. Corners are NOT in any PAD_* list -- LibreLane places them
   automatically at the 4 die corners.
""")

## 6. Rolling your own slot — a 1000 x 1000 um exercise

Imagine you want a tiny slot with 8 analog + 4 bidir + 1 vdd + 1 vss
on a square 1000 x 1000 um die. Walk through the three files:

### (a) Block in `src/slot_defines.svh`

```verilog
`ifdef SLOT_MINI
`define NUM_DVDD_PADS 1
`define NUM_DVSS_PADS 1
`define NUM_INPUT_PADS 1
`define NUM_BIDIR_PADS 4
`define NUM_ANALOG_PADS 8
`endif
```

(NUM_INPUT_PADS is 1 to avoid the Yosys zero-width-vector quirk,
not because you need a spare input.)

### (b) Arithmetic check

Total pads: 8 + 4 + 1 + 1 + 2 (clk, rst_n) + 1 (spare input) = 17.
Spread around 4 sides: roughly 4–5 per side. With 5 pads on S and
4 on the other sides:

- S side: 5 x 75 + 710 = 1085 um ... does NOT fit in 1000 um die.

So the die must grow: `DIE_AREA: [0, 0, 1200, 1200]` gives comfortable
padding. Or: fewer pads per side. Tradeoff is entirely yours.

### (c) `librelane/slots/slot_mini.yaml` (sketch)

```yaml
FP_SIZING: absolute
DIE_AREA:  [0, 0, 1200, 1200]
CORE_AREA: [250, 250, 950, 950]
VERILOG_DEFINES: ["SLOT_MINI"]

PAD_SOUTH: [
    clk_pad, rst_n_pad, "inputs\\[0\\].pad",
    "analog\\[0\\].pad", "analog\\[1\\].pad",
]
PAD_EAST: [
    "analog\\[2\\].pad", "analog\\[3\\].pad",
    "dvdd_pads\\[0\\].pad",
    "bidir\\[0\\].pad",
]
PAD_NORTH: [
    "bidir\\[2\\].pad", "bidir\\[3\\].pad",
    "analog\\[4\\].pad", "analog\\[5\\].pad",
]
PAD_WEST: [
    "analog\\[6\\].pad", "analog\\[7\\].pad",
    "dvss_pads\\[0\\].pad",
    "bidir\\[1\\].pad",
]
```

### (d) `Makefile`

Add `mini` to `AVAILABLE_SLOTS`. Done.

That's the whole recipe. No LibreLane config was changed, no
`chip_top.sv` edits, no `chip_core.sv` edits beyond writing the
RTL you want to run on the pads.

## 7. Where to go from here

Now that you understand what a slot is, the four operational
notebooks teach specific hands-on workflows. Suggested reading
order:

| notebook                             | what you learn                                                 |
|--------------------------------------|----------------------------------------------------------------|
| `rtl2gds_counter.ipynb`              | bare-block Classic flow, no padring. Warm-up.                  |
| `rtl2gds_chip_top_custom.ipynb`      | full-chip flow on an existing slot (`1x1`) with one custom macro replacing an SRAM. |
| `rtl2gds_chipathon_padring.ipynb`    | create a brand-new slot from scratch (the workshop slot). Puts sections 1–5 of this notebook into practice. |
| `rtl2gds_chipathon_use.ipynb`        | swap your own RTL into the already-provisioned workshop slot and run the full flow. |

When you tape out at the chipathon, you will mostly live inside the
fourth notebook.